# Estudio de mercado CAM – Versión **Google Maps** (Places API + Routes API)

**Objetivo:** evaluar el potencial de mercado para instalar un **CAM (Centro de Atención Médica)** en dos ubicaciones
(San Lucas y San José), considerando como zona de influencia los establecimientos a **≤ 30 minutos de traslado**
(se mide en **tiempo**, no en kilómetros).

| # | Categoría | Subtemas (color en el mapa) |
|---|---|---|
| 1 | CAM de Consulta Externa 🏥 | Clínica / centro médico · Consultorios agrupados · Consultorio adyacente a farmacia · Consultorio · Hospital (referencia) |
| 2 | Clínicas de Especialidad 🩺 | Ginecología · Pediatría · Medicina Interna · Traumatología · Cardiología · Dermatología · Oftalmología |
| 3 | Laboratorios Clínicos 🧪 | **Cadena** – Chopo · **Cadena** – Salud Digna · **Cadena** – Otra nacional · **Local** / independiente |
| 4 | Gabinetes de Imagenología 🩻 | Rayos X · Ultrasonido · Mastografía · Tomografía · Resonancia |

**Salidas por ubicación:** 5 mapas (1 general + 1 por categoría), con icono por categoría, color por subtema y
capas que se pueden encender/apagar; y un Excel con una hoja por categoría + resumen por bandas de tiempo.
Además, un Excel comparativo San Lucas vs San José.

> Datos que ninguna fuente publica de forma sistemática (número de consultorios, número de médicos) quedan
> marcados como *"validar en campo"*.

**Requisitos (Google Cloud):** una API key con **Places API (New)**, **Routes API** y **Maps JavaScript API**
habilitadas y facturación activa. Google otorga crédito mensual gratuito; este estudio (≈25 búsquedas × 3 páginas ×
2 ubicaciones + matriz de tiempos) suele quedar dentro de él, pero revisa tus cuotas.
Define la clave como variable de entorno `GOOGLE_MAPS_API_KEY` o pégala en la celda de configuración.

In [ ]:
%pip install -q requests pandas openpyxl

## 1. Configuración

In [ ]:
import os
import re, math, json, html, time, unicodedata
from pathlib import Path
from urllib.parse import unquote
import requests
import pandas as pd
GOOGLE_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "")   # o pégala aquí: "AIza..."
MODO_TRANSPORTE = "DRIVE"            # DRIVE | WALK | TWO_WHEELER | TRANSIT
CON_TRAFICO = True                   # True = TRAFFIC_AWARE (más realista, SKU Pro); False = sin tráfico (más barato)
SALIDA = Path("salidas_google"); SALIDA.mkdir(exist_ok=True)
assert GOOGLE_API_KEY, "Define GOOGLE_MAPS_API_KEY (Places API (New), Routes API y Maps JavaScript API habilitadas)"

# Ubicaciones de estudio: el enlace corto se resuelve automáticamente a coordenadas.
# Si tu red no permite resolverlo, escribe lat/lng manualmente (clic derecho en Google Maps → copiar coordenadas).
UBICACIONES = {
    "San Lucas": {"url": "https://maps.app.goo.gl/K6HAcsMj4TUccqUU7", "lat": None, "lng": None},
    "San José":  {"url": "https://maps.app.goo.gl/mVBjCMaCyxbyPQDu7", "lat": None, "lng": None},
}

TIEMPO_MAX_MIN = 30          # zona de influencia: tiempo de traslado máximo al sitio (minutos)
BANDAS_MIN = [10, 20, 30]    # bandas de tiempo para colorear / resumir
RADIO_BUSQUEDA_M = 25_000    # radio de búsqueda previo (se filtra después por TIEMPO, no por km)

## 2. Catálogo de categorías, subtemas, colores y resolución de ubicaciones

In [ ]:
def norm(txt):
    """minúsculas y sin acentos, para comparar palabras clave."""
    txt = unicodedata.normalize("NFKD", str(txt or "")).encode("ascii", "ignore").decode()
    return " " + re.sub(r"\s+", " ", txt.lower()) + " "

def contiene(texto, claves):
    """True si alguna clave aparece en el texto (los espacios en la clave marcan límite de palabra)."""
    t = norm(texto)
    return any(norm(c)[1:-1] in t for c in claves)

CAT_CAM, CAT_ESP, CAT_LAB, CAT_IMG = ("CAM Consulta Externa", "Clínica de Especialidad",
                                      "Laboratorio Clínico", "Gabinete de Imagenología")
CATEGORIAS = [CAT_CAM, CAT_ESP, CAT_LAB, CAT_IMG]

# Especialidades solicitadas (claves en español e inglés/OSM)
ESPECIALIDADES = {
    "Ginecología":       ["ginecolog", "gineco", "obstetr", "gynaecolog", "gynecolog", "maternidad"],
    "Pediatría":         ["pediatr", "paediatr", "pediatric", "ninos", "infantil"],
    "Medicina Interna":  ["medicina interna", "internista", "internal"],
    "Traumatología":     ["traumatolog", "ortoped", "orthopaed", "orthoped", "trauma"],
    "Cardiología":       ["cardiolog", "cardio", "corazon"],
    "Dermatología":      ["dermatolog", "dermato", "piel"],
    "Oftalmología":      ["oftalmolog", "ophthalmolog", "retina", "ojos"],
}

# Cadenas de laboratorio (subtema = cadena vs local)
CADENAS_LAB = {
    "Chopo":        ["chopo"],
    "Salud Digna":  ["salud digna"],
    "Otra nacional": ["olab", "laboratorio medico polanco", "lab polanco", "jenner", "laboratorios azteca",
                      "biomedica de referencia", "carpermor", " lapi ", "clinicos de puebla", "laboratorios ruiz",
                      "similares", "moreira", "diagnostico clinico hispano", "unilabs", "biolab", "labclinic",
                      "laboratorios del chopo", "orthin", "medica sur"],
}

# Estudios de imagen, del más complejo al más básico (el más complejo define el subtema)
ESTUDIOS_IMAGEN = {
    "Resonancia":   ["resonancia", " mri ", " rm "],
    "Tomografía":   ["tomograf", " tac ", "ct scan", "tomography"],
    "Mastografía":  ["mastograf", "mamograf", "mammogra"],
    "Ultrasonido":  ["ultrasonido", "ultrasound", "ecograf", "sonograf", " eco "],
    "Rayos X":      ["rayos x", "rayos-x", "x-ray", "xray", "radiograf", "radiolog"],
}
CLAVES_IMAGEN = sum(ESTUDIOS_IMAGEN.values(), []) + ["imagen", "imagenolog", "diagnostico por imagen", "imaging", "radiology"]
CLAVES_LAB = ["laborator", "analisis clinic", "lab ", "patolog", "medical_lab", "laboratory"]

SERVICIOS = {
    "Urgencias": ["urgencia", "emergenc"], "24 horas": ["24 h", "24 horas", "24/7", "open 24"],
    "Hospitalización": ["hospital"], "Farmacia": ["farmacia", "pharmacy"],
    "Laboratorio": CLAVES_LAB, "Imagen": CLAVES_IMAGEN, "Toma a domicilio": ["domicilio", "home service"],
    "Check-up": ["check up", "checkup", "chequeo"], "Consulta general": ["medicina general", "medico general"],
}

EXCLUIR = ["veterinar", "dental", "dentist", "odontolog", "ortodonc", " spa ", "estetica", "belleza", "psicolog",
           "nutriolog", "optica", "optometr", "quiropract", "fisioterap", "computo", "idiomas", "escuela", "pet "]
FARMACIAS = ["farmacia", "similares", "del ahorro", "benavides", "guadalajara", "san pablo", "farmapronto",
             "yza", "pharmacy"]

SUBTEMAS = {  # color HEX por subtema (mismo color en mapa, leyenda y Excel)
    CAT_CAM: {"Clínica / centro médico": "#2E86AB", "Consultorios agrupados / torre médica": "#1B4F72",
              "Consultorio adyacente a farmacia": "#5DADE2", "Consultorio médico": "#85C1E9",
              "Hospital (referencia, con hospitalización)": "#7F8C8D"},
    CAT_ESP: {"Ginecología": "#C2185B", "Pediatría": "#F06292", "Medicina Interna": "#8E24AA",
              "Traumatología": "#5E35B1", "Cardiología": "#D32F2F", "Dermatología": "#FF8A65",
              "Oftalmología": "#6D4C41", "Otra especialidad": "#BDBDBD"},
    CAT_LAB: {"Cadena – Chopo": "#00897B", "Cadena – Salud Digna": "#43A047",
              "Cadena – Otra nacional": "#9CCC65", "Local / independiente": "#F9A825"},
    CAT_IMG: {"Resonancia": "#212121", "Tomografía": "#546E7A", "Mastografía": "#AD1457",
              "Ultrasonido": "#0277BD", "Rayos X": "#FF6F00", "Imagen (tipo no publicado)": "#A1887F"},
}
ICONOS = {CAT_CAM: "house-medical", CAT_ESP: "user-doctor", CAT_LAB: "flask", CAT_IMG: "x-ray"}   # Font Awesome 6
EMOJIS = {CAT_CAM: "🏥", CAT_ESP: "🩺", CAT_LAB: "🧪", CAT_IMG: "🩻"}                              # Google Maps

def detectar(texto, dic):
    return [k for k, claves in dic.items() if contiene(texto, claves)]

def resolver_link(url):
    """Sigue el enlace corto de Google Maps y extrae (lat, lng)."""
    r = requests.get(url, allow_redirects=True, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    texto = unquote(r.url + " " + r.text[:300_000])
    for patron in (r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", r"@(-?\d+\.\d+),(-?\d+\.\d+)",
                   r"center=(-?\d+\.\d+),(-?\d+\.\d+)", r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)",
                   r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"):
        m = re.search(patron, texto)
        if m:
            return float(m.group(1)), float(m.group(2))
    raise ValueError(f"No se encontraron coordenadas en {r.url}")

def haversine_m(lat1, lng1, lat2, lng2):
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lng2 - lng1) * p / 2) ** 2)
    return 12_742_000 * math.asin(math.sqrt(a))

def banda(minutos):
    ini = 0
    for fin in BANDAS_MIN:
        if minutos <= fin:
            return f"{ini}-{fin} min"
        ini = fin
    return f"> {BANDAS_MIN[-1]} min"

for nombre, u in UBICACIONES.items():
    if u["lat"] is None:
        try:
            u["lat"], u["lng"] = resolver_link(u["url"])
        except Exception as e:
            raise SystemExit(f"⚠️ No pude resolver el enlace de {nombre}: {e}. Escribe lat/lng en UBICACIONES.")
    print(f"{nombre}: {u['lat']:.6f}, {u['lng']:.6f}")

## 3. Reglas de clasificación por subtema

In [ ]:
def subtema_lab(nombre):
    """(subtema, marca): la marca identifica la cadena concreta para contar sus sucursales."""
    for cadena, claves in CADENAS_LAB.items():
        for c in claves:
            if contiene(nombre, [c]):
                return f"Cadena – {cadena}", (cadena if cadena != "Otra nacional" else c.strip().title())
    return "Local / independiente", ""

def subtema_img(texto):
    estudios = detectar(texto, ESTUDIOS_IMAGEN)
    return (estudios[0] if estudios else "Imagen (tipo no publicado)"), estudios

def subtema_cam(nombre, texto):
    if contiene(nombre, ["hospital", "sanatorio"]) and not contiene(nombre, ["hospital de dia"]):
        return "Hospital (referencia, con hospitalización)"
    if contiene(nombre, FARMACIAS):
        return "Consultorio adyacente a farmacia"
    if contiene(texto, ["torre medica", "plaza medica", "consultorios", "edificio medico", "medical tower", "medical plaza"]):
        return "Consultorios agrupados / torre médica"
    if contiene(texto, ["clinica", "clinic", "centro medico", "medical center", "medical centre", "policlinica"]):
        return "Clínica / centro médico"
    return "Consultorio médico"

def clasificar(categoria, nombre, texto_extra="", pista=None):
    """Devuelve (subtema, detalle) o None si el registro no pertenece a la categoría."""
    texto = f"{nombre} {texto_extra}"
    if contiene(texto, EXCLUIR) and not contiene(texto, CLAVES_LAB + CLAVES_IMAGEN + ["clinica", "medic"]):
        return None
    if contiene(nombre, ["dental", "odontolog", "veterinar"]):
        return None
    if categoria == CAT_LAB:
        sub, cadena = subtema_lab(nombre)
        if not cadena and not contiene(texto, CLAVES_LAB):
            return None
        if contiene(nombre, ["laboratorio dental", "protesis"]):
            return None
        return sub, {"Cadena": cadena or "—"}
    if categoria == CAT_IMG:
        sub, estudios = subtema_img(texto)
        if pista and pista not in estudios:
            estudios = estudios + [pista]
            if sub.startswith("Imagen"):
                sub = pista
        if not estudios and not contiene(texto, CLAVES_IMAGEN):
            return None
        orden = list(ESTUDIOS_IMAGEN)
        estudios = sorted(set(estudios), key=orden.index)
        return (estudios[0] if estudios else sub), {"Tipos de estudio": ", ".join(estudios) or "No publicado"}
    if categoria == CAT_ESP:
        esp = detectar(texto, ESPECIALIDADES)
        if pista and pista not in esp:
            esp = [pista] + esp
        if contiene(nombre, ["hospital"]):
            return None
        sub = esp[0] if esp else "Otra especialidad"
        return sub, {"Especialidad": ", ".join(esp) or "No identificada"}
    # CAM consulta externa: descarta lo que es solo laboratorio / imagen
    if (contiene(nombre, ["laborator", "rayos x", "ultrasonido", "radiolog", "imagenolog"])
            and not contiene(nombre, ["clinica", "centro medico", "consultorio", "hospital"])):
        return None
    return subtema_cam(nombre, texto), {}

def enriquecer(df):
    """Columnas específicas que pide el estudio, calculadas por categoría."""
    if df.empty:
        return df
    df = df.copy()
    texto = (df["Nombre"].fillna("") + " " + df["Descripción"].fillna("") + " " + df["Tipos"].fillna(""))
    df["Servicios detectados"] = texto.apply(lambda t: ", ".join(detectar(t, SERVICIOS)) or "No publicado")
    df["Especialidades ofrecidas"] = texto.apply(lambda t: ", ".join(detectar(t, ESPECIALIDADES)) or "No publicado")
    labs = df[df["Categoría"] == CAT_LAB][["Latitud", "Longitud"]].values
    imgs = df[df["Categoría"] == CAT_IMG][["Latitud", "Longitud"]].values

    def en_sitio(row, puntos, claves):
        if contiene(f"{row['Nombre']} {row['Descripción']} {row['Tipos']}", claves):
            return "Sí (publicado)"
        if any(haversine_m(row["Latitud"], row["Longitud"], la, ln) < 40 for la, ln in puntos):
            return "Probable (mismo inmueble)"
        return "No detectado"
    df["Laboratorio en sitio"] = df.apply(lambda r: en_sitio(r, labs, CLAVES_LAB), axis=1)
    df["Imagen en sitio"] = df.apply(lambda r: en_sitio(r, imgs, CLAVES_IMAGEN), axis=1)
    # sucursales de la misma cadena dentro de la zona
    marca = [subtema_lab(n)[1] if c == CAT_LAB else "" for n, c in zip(df["Nombre"], df["Categoría"])]
    conteo = pd.Series(1, index=pd.MultiIndex.from_arrays([df["Ubicación estudio"], marca])).groupby(level=[0, 1]).size()
    df["Sucursales en la zona"] = [(conteo[(u, m)] if m else 1) if c == CAT_LAB else None
                                   for u, c, m in zip(df["Ubicación estudio"], df["Categoría"], marca)]
    df["Banda de tiempo"] = df["Tiempo (min)"].apply(banda)
    df["Color subtema"] = df.apply(lambda r: SUBTEMAS[r["Categoría"]].get(r["Subtema"], "#999999"), axis=1)
    return df.sort_values(["Ubicación estudio", "Categoría", "Tiempo (min)"]).reset_index(drop=True)

## 4. Búsqueda de establecimientos (Google Places) y tiempo de traslado (Google Routes)
Se busca en un radio amplio y **después se filtra por tiempo ≤ 30 min**.

In [ ]:
# (categoría, texto de búsqueda en Google, pista de subtema)
BUSQUEDAS = [
    (CAT_CAM, "clínica de consulta externa", None), (CAT_CAM, "centro médico", None),
    (CAT_CAM, "clínica médica", None),             (CAT_CAM, "consultorio médico", None),
    (CAT_CAM, "torre médica consultorios", None),  (CAT_CAM, "consultorio médico farmacia", None),
    (CAT_CAM, "hospital", None),
    (CAT_ESP, "ginecólogo", "Ginecología"),        (CAT_ESP, "pediatra", "Pediatría"),
    (CAT_ESP, "médico internista", "Medicina Interna"), (CAT_ESP, "traumatólogo ortopedista", "Traumatología"),
    (CAT_ESP, "cardiólogo", "Cardiología"),        (CAT_ESP, "dermatólogo", "Dermatología"),
    (CAT_ESP, "oftalmólogo", "Oftalmología"),
    (CAT_LAB, "laboratorio clínico", None),        (CAT_LAB, "laboratorio de análisis clínicos", None),
    (CAT_LAB, "Laboratorio Chopo", None),          (CAT_LAB, "Salud Digna", None),
    (CAT_LAB, "Olab laboratorio", None),
    (CAT_IMG, "rayos x", "Rayos X"),               (CAT_IMG, "ultrasonido", "Ultrasonido"),
    (CAT_IMG, "mastografía", "Mastografía"),       (CAT_IMG, "tomografía", "Tomografía"),
    (CAT_IMG, "resonancia magnética", "Resonancia"), (CAT_IMG, "imagenología diagnóstico por imagen", None),
]

CAMPOS = ",".join("places." + c for c in [
    "id", "displayName", "formattedAddress", "location", "types", "primaryTypeDisplayName",
    "regularOpeningHours.weekdayDescriptions", "nationalPhoneNumber", "websiteUri", "rating",
    "userRatingCount", "googleMapsUri", "editorialSummary", "businessStatus"]) + ",nextPageToken"

def buscar_places(texto, lat, lng, radio=RADIO_BUSQUEDA_M, paginas=3):
    """Places API (New) – Text Search. Hasta 60 resultados por consulta."""
    url = "https://places.googleapis.com/v1/places:searchText"
    headers = {"X-Goog-Api-Key": GOOGLE_API_KEY, "X-Goog-FieldMask": CAMPOS}
    body = {"textQuery": texto, "languageCode": "es", "regionCode": "MX", "pageSize": 20,
            "locationBias": {"circle": {"center": {"latitude": lat, "longitude": lng}, "radius": min(radio, 50_000)}}}
    salida = []
    for _ in range(paginas):
        r = requests.post(url, headers=headers, json=body, timeout=30)
        r.raise_for_status()
        data = r.json()
        salida += data.get("places", [])
        if not data.get("nextPageToken"):
            break
        body["pageToken"] = data["nextPageToken"]
        time.sleep(1)
    return salida

def tiempos_google(origenes, destino):
    """Routes API – computeRouteMatrix: tiempo desde cada establecimiento hasta el sitio del CAM."""
    url = "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
    headers = {"X-Goog-Api-Key": GOOGLE_API_KEY,
               "X-Goog-FieldMask": "originIndex,destinationIndex,duration,distanceMeters,condition"}
    wp = lambda la, ln: {"waypoint": {"location": {"latLng": {"latitude": la, "longitude": ln}}}}
    res = [(None, None)] * len(origenes)
    for ini in range(0, len(origenes), 50):             # TRAFFIC_AWARE admite 100 elementos por solicitud
        lote = origenes[ini:ini + 50]
        body = {"origins": [wp(*o) for o in lote], "destinations": [wp(*destino)], "travelMode": MODO_TRANSPORTE}
        if MODO_TRANSPORTE == "DRIVE":
            body["routingPreference"] = "TRAFFIC_AWARE" if CON_TRAFICO else "TRAFFIC_UNAWARE"
        r = requests.post(url, headers=headers, json=body, timeout=60)
        r.raise_for_status()
        for e in r.json():
            if e.get("condition") == "ROUTE_EXISTS":
                res[ini + e["originIndex"]] = (int(e["duration"].rstrip("s")) / 60, e.get("distanceMeters", 0) / 1000)
    return res

def recolectar(nombre_ubi, u):
    registros = {}
    for cat, consulta, pista in BUSQUEDAS:
        for p in buscar_places(consulta, u["lat"], u["lng"]):
            if p.get("businessStatus", "OPERATIONAL") != "OPERATIONAL" or "location" not in p:
                continue
            nombre = p.get("displayName", {}).get("text", "")
            desc = (p.get("editorialSummary") or {}).get("text", "")
            tipos = ", ".join(p.get("types", []))
            clas = clasificar(cat, nombre, f"{desc} {tipos} {p.get('primaryTypeDisplayName', {}).get('text', '')}", pista)
            if clas is None:
                continue
            la, ln = p["location"]["latitude"], p["location"]["longitude"]
            if haversine_m(u["lat"], u["lng"], la, ln) > RADIO_BUSQUEDA_M * 1.3:
                continue
            clave = (cat, p["id"])
            sub, extra = clas
            if clave in registros:                          # ya encontrado con otra búsqueda: combina estudios
                previo = registros[clave]
                if cat == CAT_IMG:
                    est = set(previo["Tipos de estudio"].split(", ")) | set(extra["Tipos de estudio"].split(", "))
                    est.discard("No publicado")
                    est = sorted(est, key=list(ESTUDIOS_IMAGEN).index)
                    previo["Tipos de estudio"], previo["Subtema"] = ", ".join(est), est[0]
                continue
            registros[clave] = {
                "Ubicación estudio": nombre_ubi, "Categoría": cat, "Subtema": sub, "Nombre": nombre,
                "Dirección": p.get("formattedAddress", ""), "Latitud": la, "Longitud": ln,
                "Horarios": " | ".join((p.get("regularOpeningHours") or {}).get("weekdayDescriptions", [])) or "No publicado",
                "Teléfono": p.get("nationalPhoneNumber", ""), "Sitio web": p.get("websiteUri", ""),
                "Calificación": p.get("rating"), "Reseñas": p.get("userRatingCount"),
                "Link mapa": p.get("googleMapsUri", ""), "Descripción": desc, "Tipos": tipos,
                "Número de consultorios": "No publicado (validar en campo)",
                "Número aproximado de médicos": "No publicado (validar en campo)",
                "Fuente": "Google Places API", **extra}
    df = pd.DataFrame(registros.values())
    if df.empty:
        return df
    t = tiempos_google(list(zip(df["Latitud"], df["Longitud"])), (u["lat"], u["lng"]))
    df["Tiempo (min)"] = [round(x[0], 1) if x[0] is not None else None for x in t]
    df["Distancia ruta (km)"] = [round(x[1], 2) if x[1] is not None else None for x in t]
    df = df[df["Tiempo (min)"].notna() & (df["Tiempo (min)"] <= TIEMPO_MAX_MIN)]
    print(f"{nombre_ubi}: {len(df)} establecimientos a ≤ {TIEMPO_MAX_MIN} min")
    return df

datos = pd.concat([recolectar(n, u) for n, u in UBICACIONES.items()], ignore_index=True)
assert not datos.empty, "No se encontraron establecimientos: revisa claves de API y coordenadas"
datos = enriquecer(datos)
datos.groupby(["Ubicación estudio", "Categoría", "Subtema"]).size().to_frame("Establecimientos")

## 5. Tablas de Excel

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Cada tabla = columnas FIJAS (ubicación) + Categoría/Subcategoría + DATOS IMPORTANTES de la categoría + referencia.
# (columna interna del DataFrame, encabezado en Excel)
FIJAS = [("Nombre", "Nombre"), ("Dirección", "Ubicación (dirección)"),
         ("Tiempo (min)", "Tiempo al punto central (min)"), ("Distancia ruta (km)", "Distancia por ruta (km)"),
         ("Banda de tiempo", "Rango de tiempo")]
CLASIFICACION = [("Categoría", "Categoría"), ("Subtema", "Subcategoría")]
IMPORTANTES = {
    CAT_CAM: [("Número de consultorios", "Número de consultorios"),
              ("Especialidades ofrecidas", "Especialidades ofrecidas"), ("Horarios", "Horarios"),
              ("Laboratorio en sitio", "Laboratorio dentro del sitio"), ("Imagen en sitio", "Imagen dentro del sitio")],
    CAT_ESP: [("Especialidad", "Especialidad"), ("Número aproximado de médicos", "Número aproximado de médicos"),
              ("Servicios detectados", "Servicios complementarios"), ("Horarios", "Horarios")],
    CAT_LAB: [("Alcance", "Alcance (cadena / independiente)"), ("Cadena", "Cadena / marca"),
              ("Sucursales en la zona", "Sucursales en la zona"), ("Horarios", "Horarios"),
              ("Servicios detectados", "Servicios más relevantes")],
    CAT_IMG: [("Tipos de estudio", "Tipos de estudio"), ("Horarios", "Horarios")],
}
REFERENCIA = [("Teléfono", "Teléfono"), ("Sitio web", "Sitio web"), ("Calificación", "Calificación Google"),
              ("Reseñas", "Núm. reseñas"), ("Latitud", "Latitud"), ("Longitud", "Longitud"),
              ("Link mapa", "Link al mapa"), ("Fuente", "Fuente")]
HOJAS = {CAT_CAM: "1_CAM_Consulta_Externa", CAT_ESP: "2_Clinicas_Especialidad",
         CAT_LAB: "3_Laboratorios", CAT_IMG: "4_Imagenologia"}
COLOR_BLOQUE = {"fijas": "1F3A5F", "clasificacion": "6A1B9A", "importantes": "2E7D32", "referencia": "616161"}

def tabla_categoria(df, cat):
    """Tabla de una categoría para un punto: fijas + categoría/subcategoría + datos importantes + referencia."""
    cols = FIJAS + CLASIFICACION + IMPORTANTES[cat] + REFERENCIA
    sub = df[df["Categoría"] == cat].copy() if not df.empty else pd.DataFrame()
    if cat == CAT_LAB and not sub.empty:
        sub["Alcance"] = sub["Subtema"].map(lambda s: "Cadena" if s.startswith("Cadena") else "Independiente")
    sub = sub.sort_values("Tiempo (min)") if not sub.empty else sub
    tabla = sub.reindex(columns=[c for c, _ in cols]).fillna("")
    tabla.columns = [e for _, e in cols]
    colores = list(sub["Color subtema"]) if not sub.empty else []
    bloques = (["fijas"] * len(FIJAS) + ["clasificacion"] * len(CLASIFICACION)
               + ["importantes"] * len(IMPORTANTES[cat]) + ["referencia"] * len(REFERENCIA))
    return tabla, colores, bloques

def _formatear(ws, df, colores=None, bloques=None):
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions
    for j, c in enumerate(ws[1]):
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor=COLOR_BLOQUE[bloques[j]] if bloques else "1F3A5F")
        c.alignment = Alignment(wrap_text=True, vertical="center")
    ws.row_dimensions[1].height = 32
    for i, col in enumerate(df.columns, 1):
        largo = max([len(str(col)) // 2] + [len(str(v)) for v in df[col].head(200)])
        ws.column_dimensions[get_column_letter(i)].width = min(max(12, largo + 2), 55)
    if colores and "Subcategoría" in df.columns:
        j = list(df.columns).index("Subcategoría") + 1
        for fila, color in enumerate(colores, 2):
            ws.cell(fila, j).fill = PatternFill("solid", fgColor=color.lstrip("#"))
            ws.cell(fila, j).font = Font(color="FFFFFF", bold=True)
    if "Link al mapa" in df.columns:
        j = list(df.columns).index("Link al mapa") + 1
        for fila in range(2, len(df) + 2):
            celda = ws.cell(fila, j)
            if str(celda.value).startswith("http"):
                celda.hyperlink, celda.value, celda.style = celda.value, "Abrir mapa", "Hyperlink"

def resumen(df):
    if df.empty:
        return pd.DataFrame()
    t = pd.pivot_table(df, index=["Categoría", "Subtema"], columns="Banda de tiempo", values="Nombre",
                       aggfunc="count", fill_value=0)
    t["Total"] = t.sum(axis=1)
    t["Tiempo mínimo (min)"] = df.groupby(["Categoría", "Subtema"])["Tiempo (min)"].min().round(1)
    return t.reset_index().rename(columns={"Subtema": "Subcategoría"})

def exportar_excel(df, ruta, parametros):
    """Un archivo por punto: 4 tablas (una por categoría) + resumen + metodología."""
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        for cat in CATEGORIAS:
            tabla, colores, bloques = tabla_categoria(df, cat)
            tabla.to_excel(xw, sheet_name=HOJAS[cat], index=False)
            _formatear(xw.sheets[HOJAS[cat]], tabla, colores, bloques)
        r = resumen(df)
        r.to_excel(xw, sheet_name="Resumen", index=False)
        _formatear(xw.sheets["Resumen"], r)
        met = pd.DataFrame(parametros.items(), columns=["Parámetro", "Valor"])
        met.to_excel(xw, sheet_name="Metodologia", index=False)
        _formatear(xw.sheets["Metodologia"], met)
    print("Excel generado:", ruta)

def exportar_comparativo(df, ruta):
    if df.empty:
        return
    t = pd.pivot_table(df, index=["Categoría", "Subtema"], columns="Ubicación estudio", values="Nombre",
                       aggfunc="count", fill_value=0).reset_index().rename(columns={"Subtema": "Subcategoría"})
    m = pd.pivot_table(df, index="Categoría", columns="Ubicación estudio", values="Tiempo (min)",
                       aggfunc=["count", "min", "median"]).round(1)
    m.columns = [f"{a} – {b}".replace("count", "Total").replace("min", "Tiempo mín").replace("median", "Tiempo mediana")
                 for a, b in m.columns]
    m = m.reset_index()
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        t.to_excel(xw, sheet_name="Comparativo_subcategorias", index=False)
        _formatear(xw.sheets["Comparativo_subcategorias"], t)
        m.to_excel(xw, sheet_name="Comparativo_categorias", index=False)
        _formatear(xw.sheets["Comparativo_categorias"], m)
    print("Excel comparativo generado:", ruta)

In [ ]:
PARAMETROS = {
    "Fuente de establecimientos": "Google Places API (New) – Text Search",
    "Tiempo de traslado": f"Google Routes API – computeRouteMatrix, modo {MODO_TRANSPORTE}, "
                          f"{'con tráfico (TRAFFIC_AWARE, hora de ejecución)' if CON_TRAFICO else 'sin tráfico'}",
    "Sentido del traslado": "Desde cada establecimiento hacia el sitio propuesto del CAM",
    "Zona de influencia": f"≤ {TIEMPO_MAX_MIN} minutos (bandas {BANDAS_MIN})",
    "Fecha de extracción": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "Número de consultorios / médicos": "Google no los publica: columnas marcadas para validación en campo",
    "Subtemas": "Clasificación automática por nombre, tipo y descripción; revisar casos dudosos",
}
for nombre_ubi in UBICACIONES:
    u = UBICACIONES[nombre_ubi]
    p = dict(PARAMETROS, **{"Ubicación": f"{nombre_ubi} ({u['lat']:.6f}, {u['lng']:.6f})", "Enlace": u["url"]})
    exportar_excel(datos[datos["Ubicación estudio"] == nombre_ubi], SALIDA / f"CAM_{norm(nombre_ubi).strip().replace(' ', '_')}_google.xlsx", p)
exportar_comparativo(datos, SALIDA / "CAM_comparativo_google.xlsx")

## 6. Mapas en Google Maps (5 por ubicación)
Se generan archivos HTML con Maps JavaScript API: ★ = sitio propuesto; el **emoji** del pin identifica la categoría y el **color** el subtema. La leyenda permite ocultar/mostrar cada subtema. Ábrelos en el navegador si el visor de Jupyter no muestra el mapa (las restricciones de *referrer* de la API key pueden bloquear el iframe).

In [ ]:
from IPython.display import IFrame, display

PLANTILLA = """<!doctype html><html><head><meta charset="utf-8"><title>__TITULO__</title>
<style>html,body,#map{height:100%;margin:0;font-family:Arial,sans-serif}
#leyenda{position:absolute;top:10px;right:10px;background:#fff;padding:10px 12px;border-radius:8px;
 box-shadow:0 2px 6px rgba(0,0,0,.3);max-height:85%;overflow:auto;font-size:13px;z-index:5}
#leyenda h4{margin:6px 0 4px}.sw{display:inline-block;width:12px;height:12px;border-radius:50%;margin-right:4px}
</style></head><body><div id="map"></div><div id="leyenda"><b>__TITULO__</b><br>
<small>Zona: ≤ __TMAX__ min de traslado · clic en la casilla para ocultar/mostrar</small><div id="items"></div></div>
<script>
const SITIO = __SITIO__, PUNTOS = __PUNTOS__, COLORES = __COLORES__, EMOJIS = __EMOJIS__;
async function initMap() {
  const {Map, InfoWindow} = await google.maps.importLibrary("maps");
  const {AdvancedMarkerElement, PinElement} = await google.maps.importLibrary("marker");
  const map = new Map(document.getElementById("map"), {center: SITIO, zoom: 12, mapId: "DEMO_MAP_ID"});
  const info = new InfoWindow();
  const sitio = new PinElement({background: "#E53935", borderColor: "#7f0000", glyph: "★", glyphColor: "#fff", scale: 1.6});
  new AdvancedMarkerElement({map, position: SITIO, content: sitio.element, title: "Sitio propuesto CAM"});
  const grupos = {};
  for (const p of PUNTOS) {
    const pin = new PinElement({background: p.color, borderColor: "#333", glyph: EMOJIS[p.cat], scale: 1.1});
    const m = new AdvancedMarkerElement({map, position: {lat: p.lat, lng: p.lng}, content: pin.element, title: p.nombre});
    m.addListener("click", () => { info.setContent(p.popup); info.open({map, anchor: m}); });
    const k = p.cat + "||" + p.sub; (grupos[k] = grupos[k] || []).push(m);
  }
  const cont = document.getElementById("items"); let catPrev = "";
  for (const k of Object.keys(grupos).sort()) {
    const [cat, sub] = k.split("||");
    if (cat !== catPrev) { cont.insertAdjacentHTML("beforeend", `<h4>${EMOJIS[cat]} ${cat}</h4>`); catPrev = cat; }
    const id = "c" + Math.random().toString(36).slice(2);
    cont.insertAdjacentHTML("beforeend", `<label><input type="checkbox" id="${id}" checked>
      <span class="sw" style="background:${COLORES[k]}"></span>${sub} (${grupos[k].length})</label><br>`);
    document.getElementById(id).onchange = e => grupos[k].forEach(m => m.map = e.target.checked ? map : null);
  }
}
</script>
<script async src="https://maps.googleapis.com/maps/api/js?key=__KEY__&v=weekly&libraries=marker&callback=initMap"></script>
</body></html>"""

def popup(r):
    campos = ["Categoría", "Subtema", "Dirección", "Tiempo (min)", "Horarios", "Teléfono"]
    campos += [c for c, _ in IMPORTANTES[r["Categoría"]] if c not in campos and c in r]
    filas = "".join(f"<tr><td><b>{c}</b></td><td>{html.escape(str(r.get(c, '')))}</td></tr>" for c in campos)
    return f"<div style='max-width:320px'><h4>{html.escape(r['Nombre'])}</h4><table>{filas}</table>" \
           f"<a href='{r['Link mapa']}' target='_blank'>Ver en Google Maps</a></div>"

def mapa_google(nombre_ubi, df, titulo, archivo):
    u = UBICACIONES[nombre_ubi]
    puntos = [{"lat": r["Latitud"], "lng": r["Longitud"], "cat": r["Categoría"], "sub": r["Subtema"],
               "color": r["Color subtema"], "nombre": r["Nombre"], "popup": popup(r)} for _, r in df.iterrows()]
    colores = {f"{c}||{s}": col for c, subs in SUBTEMAS.items() for s, col in subs.items()}
    doc = (PLANTILLA.replace("__TITULO__", html.escape(titulo)).replace("__TMAX__", str(TIEMPO_MAX_MIN))
           .replace("__SITIO__", json.dumps({"lat": u["lat"], "lng": u["lng"]}))
           .replace("__PUNTOS__", json.dumps(puntos, ensure_ascii=False))
           .replace("__COLORES__", json.dumps(colores, ensure_ascii=False))
           .replace("__EMOJIS__", json.dumps(EMOJIS, ensure_ascii=False)).replace("__KEY__", GOOGLE_API_KEY))
    ruta = SALIDA / archivo
    ruta.write_text(doc, encoding="utf-8")
    return ruta

# 5 mapas por ubicación: 1 general + 1 por cada categoría del estudio
for nombre_ubi in UBICACIONES:
    d = datos[datos["Ubicación estudio"] == nombre_ubi]
    slug = norm(nombre_ubi).strip().replace(" ", "_")
    rutas = [mapa_google(nombre_ubi, d, f"{nombre_ubi} – Todos los establecimientos", f"{slug}_0_general.html")]
    for i, cat in enumerate(CATEGORIAS, 1):
        rutas.append(mapa_google(nombre_ubi, d[d["Categoría"] == cat], f"{nombre_ubi} – {cat}", f"{slug}_{i}_{norm(cat).split()[0]}.html"))
    print("\n".join(map(str, rutas)))
    display(IFrame(str(rutas[0]), width="100%", height=600))

## 7. Las 4 tablas por punto (las mismas que van al Excel)

In [ ]:

for nombre_ubi in UBICACIONES:
    for cat in CATEGORIAS:
        tabla, _, _ = tabla_categoria(datos[datos["Ubicación estudio"] == nombre_ubi], cat)
        print(f"\n=== {nombre_ubi} · {cat} ({len(tabla)}) ===")
        display(tabla)
